<a href="https://colab.research.google.com/github/dprobity/machinelearning/blob/main/Copy_of_CS583HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This part runs the arithemetic and keysequence data generation

In [ ]:
import pandas as pd
import random

# Generate Arithmetic Operations sequences with two patterns
arithmetic_sequences = []

for i in range(100):
    pattern_type = i % 2  # Define 2 types of patterns
    sequence = []
    if pattern_type == 0:  # Alternating differences (e.g., +5, -3, +5, -3)
        start = random.randint(500, 1000)
        diff1, diff2 = random.randint(1, 10), random.randint(-10, -1)
        sequence = [start]
        while len(sequence) < 100:
            next_value = sequence[-1] + (diff1 if len(sequence) % 2 == 1 else diff2)
            if next_value <= 0:
                next_value = random.randint(1, 100)  # Reset to positive if needed
            sequence.append(next_value)
    elif pattern_type == 1:  # Repeating fixed difference (e.g., +7, +7, +7, ...)
        start = random.randint(500, 1000)
        diff = random.randint(1, 20)
        sequence = [start]
        for _ in range(99):
            next_value = sequence[-1] + diff
            if next_value <= 0:
                next_value = random.randint(1, 100)  # Reset to positive if needed
            sequence.append(next_value)
    arithmetic_sequences.append(sequence)

# Generate Key Sequences with two patterns
keyboard_keys = ['Q', 'W', 'E', 'R', 'T', 'Y']
key_sequences = []

for i in range(100):
    pattern_type = i % 2  # Define 2 types of patterns
    sequence = []
    if pattern_type == 0:  # Alternating pattern
        sequence = [keyboard_keys[j % len(keyboard_keys)] for j in range(100)]
    elif pattern_type == 1:  # Gradual shifts
        start_index = random.randint(0, len(keyboard_keys) - 1)
        sequence = [keyboard_keys[(start_index + j) % len(keyboard_keys)] for j in range(100)]
    key_sequences.append(sequence)

# Create DataFrames
arithmetic_df = pd.DataFrame(arithmetic_sequences).add_prefix("Arithmetic_")
key_df = pd.DataFrame(key_sequences).add_prefix("Key_Sequences_")

# Save to CSV
arithmetic_df.to_csv("Patterned_Arithmetic_Sequences.csv", index=False)
key_df.to_csv("Patterned_Key_Sequences.csv", index=False)

print("Datasets saved as 'Arithmetic_Sequences.csv' and 'Key_Sequences.csv'.")

Datasets saved as 'Arithmetic_Sequences.csv' and 'Key_Sequences.csv'.


# **It makes sense to see what the generated raw data looks like before preprocessing it to be fed to the network**

In [ ]:
# Show raw arithmetic sequences
print("Raw Arithmetic Sequences head:")
display(arith_df.head())

print("Raw Arithmetic Sequences tail:")

display(arith_df.tail())


# Show raw key sequences
print("Raw Key Sequences data head:")
display(keys_df.head())


print("Raw Key Sequences data tail:")

display(keys_df.tail())


Raw Arithmetic Sequences head:


NameError: name 'arith_df' is not defined

## **## To make the codes easier to read, i have made it modular: here i have defined fuctions for the different bits and there is the last cell which calls the functions to do the training and visulaization**

and the error handling is within each function block to track origins of error in the code execution pipeline

In [ ]:
# =========================
# Imports
# =========================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt


In [ ]:
def load_datasets(arith_path="Patterned_Arithmetic_Sequences.csv",
                  key_path="Patterned_Key_Sequences.csv"):
    try:
        arith = pd.read_csv(arith_path)
        keys = pd.read_csv(key_path)
        print("Datasets loaded successfully.")
        return arith, keys
    except Exception as e:
        print("Error loading datasets:", e)
        raise


In [ ]:
def preprocess_arithmetic(arith_df, window=10):
    try:
        scaler = MinMaxScaler()
        arith_scaled = scaler.fit_transform(arith_df)

        X, y = [], []
        for seq in arith_scaled:
            for i in range(len(seq) - window):
                X.append(seq[i:i+window])
                y.append(seq[i+window])
        X = np.array(X).reshape(-1, window, 1)
        y = np.array(y)
        print("Arithmetic preprocessing complete.")
        return X, y, scaler
    except Exception as e:
        print("Error in arithmetic preprocessing:", e)
        raise


def preprocess_keys(keys_df, window=10):
    try:
        encoder = LabelEncoder()
        keys_encoded = keys_df.apply(encoder.fit_transform)

        X, y = [], []
        for seq in keys_encoded.values:
            for i in range(len(seq) - window):
                X.append(seq[i:i+window])
                y.append(seq[i+window])
        X = np.array(X).reshape(-1, window, 1)
        y = to_categorical(y, num_classes=len(encoder.classes_))
        print("Key preprocessing complete.")
        return X, y, encoder
    except Exception as e:
        print("Error in key preprocessing:", e)
        raise


In [ ]:
def build_arithmetic_model(input_shape):
    try:
        model = Sequential([
            SimpleRNN(50, activation='relu', input_shape=input_shape),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        print("Arithmetic model built.")
        return model
    except Exception as e:
        print("Error building arithmetic model:", e)
        raise


def build_key_model(input_shape, num_classes):
    try:
        model = Sequential([
            SimpleRNN(50, activation='relu', input_shape=input_shape),
            Dense(num_classes, activation='softmax')
        ])
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        print("Key model built.")
        return model
    except Exception as e:
        print("Error building key model:", e)
        raise


In [ ]:
def train_model(model, X_train, y_train, epochs=200, batch_size=64, val_split=0.1):
    try:
        history = model.fit(X_train, y_train,
                            validation_split=val_split,
                            epochs=epochs,
                            batch_size=batch_size,
                            verbose=1)
        print("Training complete.")
        return history
    except Exception as e:
        print("Error during training:", e)
        raise


In [ ]:
def plot_loss(history, title="Training vs Validation Loss"):
    try:
        plt.plot(history.history['loss'], label='Train Loss')
        if 'val_loss' in history.history:
            plt.plot(history.history['val_loss'], label='Val Loss')
        plt.title(title)
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()
    except Exception as e:
        print("Error plotting loss:", e)
        raise


In [ ]:
def evaluate_model(model, X_test, y_test, task="Arithmetic"):
    try:
        results = model.evaluate(X_test, y_test, verbose=0)
        print(f"{task} Test Results:", results)
        return results
    except Exception as e:
        print("Error during evaluation:", e)
        raise


In [ ]:
# Load data
arith_df, keys_df = load_datasets()

# Preprocess
Xa, ya, scaler = preprocess_arithmetic(arith_df)
Xk, yk, encoder = preprocess_keys(keys_df)

# Train/test split
Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.2)
Xk_train, Xk_test, yk_train, yk_test = train_test_split(Xk, yk, test_size=0.2)

# Build models
arith_model = build_arithmetic_model((Xa_train.shape[1], 1))
key_model = build_key_model((Xk_train.shape[1], 1), len(encoder.classes_))

# Train
hist_arith = train_model(arith_model, Xa_train, ya_train)
hist_keys = train_model(key_model, Xk_train, yk_train)

# Visualize
plot_loss(hist_arith, "Arithmetic Loss")
plot_loss(hist_keys, "Key Sequences Loss")

# Evaluate
evaluate_model(arith_model, Xa_test, ya_test, "Arithmetic")
evaluate_model(key_model, Xk_test, yk_test, "Keys")
